In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import re
import unicodedata

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import (
    classification_report,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    accuracy_score
)

from xgboost import XGBClassifier

In [ ]:
# ============================================================
# 2. LOAD DATA
# ============================================================

# Assumes raw_data_df already exists.
# Otherwise, uncomment and adjust:

raw_data_df = pd.read_excel(
    r"C:\Oren\Work\CBS\DUMMY_DATASET.xlsx"
)

df = raw_data_df.copy()

print("Original dataset shape:", df.shape)

In [ ]:
# ============================================================
# 3. DEFINE TARGET AND FEATURES
# ============================================================

target_col = "SemelMishlachSofi"

text_cols = [
    "ShemAvoda",
    "SugAvoda",
    "ShemMachlaka",
    "SugMachlaka",
    "EzoAvoda",
    "TeurPeula",
    "TeurTafkid"
]

cat_cols = [
    "YeshuvAvoda",
    "MenahelEtMi",
    "MaamadAvoda",
    "MakorSachar",
    "TeudaGvoha",
    "SemelAnafSofi"
]

num_cols = [
    "shnotlimud",
    "Gil"
]

# Keep only columns that exist in the dataset.

text_cols = [c for c in text_cols if c in df.columns]
cat_cols = [c for c in cat_cols if c in df.columns]
num_cols = [c for c in num_cols if c in df.columns]

# Extract employment starting year.

if "TarichHathala" in df.columns:

    dates = pd.to_datetime(
        df["TarichHathala"],
        errors="coerce",
        dayfirst=True
    )

    df["StartYear"] = dates.dt.year

    num_cols.append("StartYear")


print("\nText features:", text_cols)
print("Categorical features:", cat_cols)
print("Numerical features:", num_cols)


# ============================================================
# 4. CLEAN TARGET
# ============================================================

df[target_col] = (
    df[target_col]
    .astype("string")
    .str.strip()
)

# Remove missing target values.

df = df.loc[
    df[target_col].notna()
].copy()

# Remove empty target values.

df = df.loc[
    df[target_col] != ""
].copy()

# Remove incomplete occupational codes containing X.

df = df.loc[
    ~df[target_col].str.contains(
        "X",
        case=False,
        na=False
    )
].copy()

print("\nAfter target cleaning:")
print("Dataset shape:", df.shape)


# ============================================================
# 5. REMOVE RARE TARGET CLASSES
# ============================================================

MIN_CLASS_SIZE = 5

class_counts = df[target_col].value_counts()

valid_classes = class_counts[
    class_counts >= MIN_CLASS_SIZE
].index

df = df.loc[
    df[target_col].isin(valid_classes)
].copy()

print("\nAfter removing rare classes:")
print("Dataset shape:", df.shape)

print("Number of classes:", df[target_col].nunique())

print("\nClass distribution:")
print(df[target_col].value_counts())


# ============================================================
# 6. CLEAN TEXT FEATURES
# ============================================================

def clean_text(value):

    if pd.isna(value):
        return ""

    value = str(value)

    # Unicode normalization.

    value = unicodedata.normalize(
        "NFC",
        value
    )

    # Remove excessive whitespace.

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


for col in text_cols:

    df[col] = df[col].apply(clean_text)


# ============================================================
# 7. COMBINE TEXT FEATURES
# ============================================================

def combine_text(row):

    parts = []

    for col in text_cols:

        value = row[col]

        if value:

            parts.append(
                f"{col}: {value}"
            )

    return " ".join(parts)


df["text_all"] = df[text_cols].apply(
    combine_text,
    axis=1
)

print("\nExample combined text:")
print(df["text_all"].iloc[0])


# ============================================================
# 8. CLEAN CATEGORICAL FEATURES
# ============================================================

# Treat incomplete industry codes as missing.

if "SemelAnafSofi" in cat_cols:

    df["SemelAnafSofi"] = df[
        "SemelAnafSofi"
    ].replace(
        ["XXXX", "1XXX"],
        np.nan
    )


# Replace all missing categorical values with MISSING.
# Convert to ordinary Python strings.

for col in cat_cols:

    df[col] = (
        df[col]
        .astype("object")
        .where(
            df[col].notna(),
            "MISSING"
        )
        .astype(str)
    )


# ============================================================
# 9. CLEAN NUMERICAL FEATURES
# ============================================================

for col in num_cols:

    # Convert values to numeric.
    # Invalid values become missing.

    numeric_series = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    # Convert pandas nullable values (pd.NA)
    # explicitly into NumPy np.nan.

    df[col] = numeric_series.to_numpy(
        dtype=np.float64,
        na_value=np.nan
    )

    # Guarantee ordinary NumPy float64 dtype.

    df[col] = df[col].astype(np.float64)


print("\nNumerical data types:")
print(df[num_cols].dtypes)

print("\nMissing numerical values:")
print(df[num_cols].isna().sum())

In [ ]:
# ============================================================
# 10. DEFINE X AND y
# ============================================================

feature_cols = (
    ["text_all"] +
    cat_cols +
    num_cols
)

X = df[feature_cols].copy()

y = df[target_col].astype(str).copy()


print("\nFeature matrix:", X.shape)
print("Target vector:", y.shape)


# ============================================================
# 11. VALIDATE INPUT DATA TYPES
# ============================================================

print("\nFeature data types:")
print(X.dtypes)

# Check that no pandas NA values remain.

for col in cat_cols:

    assert not X[col].isna().any(), (
        f"Missing categorical values in {col}"
    )

for col in num_cols:

    assert X[col].dtype == np.float64, (
        f"Unexpected numerical dtype in {col}"
    )

    # Numerical missing values must be np.nan.

    assert not pd.isna(X[col]).any() or (
        X[col].isna().any()
    )

print("\nInput validation completed.")


# ============================================================
# 12. ENCODE TARGET
# ============================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

num_classes = len(label_encoder.classes_)

print("\nNumber of target classes:", num_classes)

print("\nTarget encoding:")

for i, label in enumerate(label_encoder.classes_):

    print(i, "->", label)


# ============================================================
# 13. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y_encoded,

    test_size=0.20,

    random_state=42,

    stratify=y_encoded

)

print("\nTraining observations:", len(X_train))
print("Testing observations:", len(X_test))

In [ ]:
# ============================================================
# 14. TF-IDF PIPELINE
# ============================================================

text_pipeline = Pipeline([

    (
        "tfidf",

        TfidfVectorizer(

            analyzer="word",

            ngram_range=(1, 2),

            min_df=2,

            max_df=0.95,

            max_features=500,

            sublinear_tf=True,

            lowercase=False

        )
    )

])


# ============================================================
# 15. CATEGORICAL PIPELINE
# ============================================================

# Missing categorical values have already been replaced.

categorical_pipeline = Pipeline([

    (
        "encoder",

        OneHotEncoder(

            handle_unknown="ignore",

            sparse_output=True

        )
    )

])


# ============================================================
# 16. NUMERICAL PIPELINE
# ============================================================

numerical_pipeline = Pipeline([

    (
        "imputer",

        SimpleImputer(

            strategy="median",

            missing_values=np.nan,

            add_indicator=True

        )
    )

])


# ============================================================
# 17. COMBINE PREPROCESSING PIPELINES
# ============================================================

transformers = [

    (
        "text",

        text_pipeline,

        "text_all"

    )

]


if cat_cols:

    transformers.append(

        (
            "categorical",

            categorical_pipeline,

            cat_cols

        )

    )


if num_cols:

    transformers.append(

        (
            "numerical",

            numerical_pipeline,

            num_cols

        )

    )


preprocessor = ColumnTransformer(

    transformers=transformers,

    remainder="drop",

    sparse_threshold=1.0

)


# ============================================================
# 18. TRANSFORM TRAINING AND TEST DATA
# ============================================================

print("\nFitting preprocessing pipelines...")

X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

print("\nPreprocessing completed.")

print(
    "Training matrix:",
    X_train_processed.shape
)

print(
    "Testing matrix:",
    X_test_processed.shape
)

In [ ]:
# ============================================================
# 19. TRAIN XGBOOST CLASSIFIER
# ============================================================

model = XGBClassifier(

    objective="multi:softprob",

    num_class=num_classes,

    n_estimators=150,

    max_depth=2,

    learning_rate=0.05,

    min_child_weight=2,

    subsample=0.8,

    colsample_bytree=0.8,

    reg_lambda=5,

    eval_metric="mlogloss",

    tree_method="hist",

    n_jobs=1,

    random_state=42

)


print("\nTraining XGBoost...")

model.fit(
    X_train_processed,
    y_train
)

print("Training completed.")


# ============================================================
# 20. PREDICTIONS
# ============================================================

y_pred = model.predict(
    X_test_processed
)

# Convert encoded labels back to original codes.

y_test_original = label_encoder.inverse_transform(
    y_test
)

y_pred_original = label_encoder.inverse_transform(
    y_pred.astype(int)
)


# ============================================================
# 21. MODEL EVALUATION
# ============================================================

print("\n========================================")
print("MODEL EVALUATION")
print("========================================")

print(
    "\nAccuracy:",
    round(
        accuracy_score(y_test, y_pred),
        4
    )
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_test,
            y_pred,
            average="macro"
        ),
        4
    )
)

print(
    "Weighted F1:",
    round(
        f1_score(
            y_test,
            y_pred,
            average="weighted"
        ),
        4
    )
)

print(
    "Balanced Accuracy:",
    round(
        balanced_accuracy_score(
            y_test,
            y_pred
        ),
        4
    )
)


print("\nClassification Report:\n")

print(
    classification_report(

        y_test,

        y_pred,

        labels=np.arange(num_classes),

        target_names=label_encoder.classes_,

        zero_division=0

    )
)


# ============================================================
# 22. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(

    y_test,

    y_pred,

    labels=np.arange(num_classes)

)


confusion_df = pd.DataFrame(

    cm,

    index=label_encoder.classes_,

    columns=label_encoder.classes_

)


print("\nConfusion Matrix:")

display(confusion_df)


# ============================================================
# 23. PREDICTION RESULTS
# ============================================================

results_df = pd.DataFrame({

    "Actual_Code": y_test_original,

    "Predicted_Code": y_pred_original,

    "Correct": (
        y_test_original == y_pred_original
    )

})


print("\nPrediction Results:")

display(results_df)


# ============================================================
# 24. FEATURE IMPORTANCE
# ============================================================

importance = model.feature_importances_

feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({

    "Feature": feature_names,

    "Importance": importance

})


importance_df = importance_df.sort_values(

    by="Importance",

    ascending=False

)


print("\nTop 30 Features:")

display(
    importance_df.head(30)
)


# ============================================================
# 25. FINAL SUMMARY
# ============================================================

print("\n========================================")
print("FINAL SUMMARY")
print("========================================")

print("Original observations:", len(raw_data_df))

print("Observations used:", len(df))

print("Number of classes:", num_classes)

print("Number of text features:", len(text_cols))

print("Number of categorical features:", len(cat_cols))

print("Number of numerical features:", len(num_cols))

print(
    "Number of transformed features:",
    X_train_processed.shape[1]
)

print(
    "Test accuracy:",
    round(accuracy_score(y_test, y_pred), 4)
)

print(
    "Test macro-F1:",
    round(
        f1_score(
            y_test,
            y_pred,
            average="macro"
        ),
        4
    )
)